# Gemma4EC — Predict EC Numbers

This notebook walks you through predicting EC numbers from reaction SMILES using Gemma4EC.

## Steps
1. Reaction SMILES are formatted into prompts
2. A pretrained LLM (TxGemma) is loaded from Hugging Face
3. A LoRA adapter trained for EC prediction is loaded from the Hugging Face Hub
4. The model generates EC number predictions for each reaction

## Before running
- [ ] You have the `gemma4ec` conda environment activated
- [ ] You have requested access to TxGemma on Hugging Face ([9B](https://huggingface.co/google/txgemma-9b-predict) or [2B](https://huggingface.co/google/txgemma-2b-predict))
- [ ] You have a GPU with ≥18 GB VRAM (9B model) or ≥6 GB (2B model)

> **The Gemma4EC adapter (`PlanesLab/Gemma4EC-9B-Predict`) loads automatically from the Hugging Face Hub — no manual download needed.**

## 1. Authenticate with Hugging Face

TxGemma is a gated model — you need a Hugging Face account and to have requested access.
Run this once and your token will be saved locally (you won't need to do this again).

In [ ]:
import os

# huggingface_hub reads HF_TOKEN automatically — no login() call needed,
# which avoids writing to ~/.cache/huggingface/ (home dir disk quota issue).
_hf_token = os.environ.get("HUGGINGFACE_TOKEN", "") or os.environ.get("HF_TOKEN", "")
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    print("HF_TOKEN set from environment variable — authentication ready.")
else:
    print("WARNING: No HF token found in environment.")
    print("  Set it before launching Jupyter:  export HUGGINGFACE_TOKEN=hf_...")
    print("  Gated models (txgemma-9b-predict) will fail to download.")

## 2. Configure paths

These defaults match the repo structure — **you only need to change `ADAPTER_FOLDER`** to point to where you downloaded the adapter.
If you put it in `models/Gemma4EC-9B-Predict` as suggested in the README, you don't need to change anything.

In [ ]:
import gc
import os
import sys
from pathlib import Path

# ── Resolve repo root ─────────────────────────────────────────────────────────
# Works whether Jupyter was launched from the repo root or from notebooks/.
_HERE = Path(".").resolve()
_REPO = _HERE.parent if _HERE.name == "notebooks" else _HERE

if not (_REPO / "prompts" / "prompts.json").exists():
    raise FileNotFoundError(
        f"Cannot find prompts/prompts.json under {_REPO}.\n"
        "Make sure you launched Jupyter from the repo root or notebooks/ directory."
    )

# Add repo root to sys.path so 'src.*' imports work
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

# Set HF cache to scratch (not home dir, which has quota)
_HF_CACHE = _REPO / ".cache/gemma4ec/hf"
_HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(_HF_CACHE))

# ── Only change these if your setup differs from the defaults ─────────────────

ADAPTER_FOLDER   = "PlanesLab/Gemma4EC-9B-Predict"       # HF Hub — no download needed
# ADAPTER_FOLDER = str(_REPO / "models/Gemma4EC-9B-Predict")  # LOCAL alternative

MODEL_ID         = "google/txgemma-9b-predict"            # or txgemma-2b-predict

INPUT_CSV        = str(_HERE / "reactions.csv")
OUTPUT_DIR       = str(_HERE / "outputs/results")
PROCESSED_DIR    = str(_HERE / "outputs/inter_data")
PROMPTS_JSON     = str(_REPO / "prompts/prompts.json")
CACHE_DIR        = str(_HF_CACHE)
SAVE_DIR         = str(_REPO / ".cache/gemma4ec/models")

# ─────────────────────────────────────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

print("✓ Paths configured")
print(f"  Repo root : {_REPO}")
print(f"  Adapter   : {ADAPTER_FOLDER}")
print(f"  Model     : {MODEL_ID}")
print(f"  Prompts   : {PROMPTS_JSON}")
print(f"  Output    : {OUTPUT_DIR}")

## 3. Prepare input reactions

Your input must be a CSV with a `rxn_smiles` column.
The cell below creates a small example — replace it with `pd.read_csv('your_file.csv')` to use your own data.

In [ ]:
import pandas as pd

# ── Option A: use example reactions ───────────────────────────────────────────
df = pd.DataFrame({
    "rxn_smiles": [
        "OC(=O)CCC(=O)C(=O)O.OC(=O)CCC(N)C(=O)O>>OC(=O)CCC(=O)C(=O)O",
        "CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(O)(=O)OP(O)(=O)OCC1OC(n2cnc3c(N)ncnc23)C(O)C1O>>CC(O)C(C)(C)COP(O)(=O)OP(O)(=O)OCC1OC(n2cnc3c(N)ncnc23)C(O)C1O",
    ]
})

# ── Option B: load your own CSV ───────────────────────────────────────────────
# df = pd.read_csv("path/to/your_reactions.csv")
# assert "rxn_smiles" in df.columns, "CSV must have a 'rxn_smiles' column"

df.to_csv(INPUT_CSV, index=False)
print(f"✓ {len(df)} reaction(s) ready")
df

## 4. Format reactions into prompts

Gemma4EC expects reactions wrapped in a specific prompt template.
This step converts your CSV into a JSONL file with the correct format.

In [ ]:
from src.datamod.templates import build_input_template_from_file
from src.datamod.jsonl_builder import create_jsonl_from_df
from src.utils.paths import get_jsonl_path

TEMPLATE_KEY   = "EC_NUMBER_PROMPT_2"   # recommended — do not change unless retraining
TEMPLATE_ALIAS = "EC_NUMBER"

input_template = build_input_template_from_file(
    template_path=PROMPTS_JSON,
    key=TEMPLATE_KEY,
    required_vars={"reaction_smiles"},
    dict_key=TEMPLATE_ALIAS,
)

jsonl_path = get_jsonl_path(PROCESSED_DIR, "predict", TEMPLATE_ALIAS, TEMPLATE_KEY)
create_jsonl_from_df(INPUT_CSV, jsonl_path, input_template)

print(f"✓ Prompts written to: {jsonl_path}")

## 5. Load the model and adapter

This step downloads TxGemma from Hugging Face (first run only — ~18 GB for 9B, ~4 GB for 2B)
and loads the LoRA adapter on top.

> **This will take a few minutes on first run** while the model downloads. Subsequent runs load from cache instantly.

In [ ]:
import gc
import os
import torch
from pathlib import Path

os.environ.setdefault("WORLD_SIZE", "1")
os.environ.setdefault("RANK", "0")
os.environ.setdefault("LOCAL_RANK", "0")
os.environ.setdefault("MASTER_ADDR", "localhost")
os.environ.setdefault("MASTER_PORT", "12355")

from accelerate import Accelerator
from peft import PeftModel
from src.models.modeling import build_model_and_tokenizer
from src.training.manual.checkpoint_utils import reload_best_checkpoint

# free any leftover from a previous run of this cell
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except NameError:
    pass

accelerator = Accelerator()

print(f"Loading base model: {MODEL_ID} ...")
model, tokenizer = build_model_and_tokenizer(MODEL_ID, SAVE_DIR, CACHE_DIR)

print(f"Loading adapter: {ADAPTER_FOLDER} ...")
if os.path.isdir(ADAPTER_FOLDER):
    # Local path — use the custom loader (handles in-place reload for DDP)
    model = reload_best_checkpoint(model, accelerator, ADAPTER_FOLDER)
else:
    # HF Hub ID — download via huggingface_hub first, then load from local path.
    # This avoids a peft bug (deprecated `use_auth_token` kwarg) that causes a
    # TypeError when peft tries to fetch the config directly from HF Hub.
    from huggingface_hub import snapshot_download
    _hf_hub_cache = os.path.join(
        os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface")), "hub"
    )
    _adapter_local = snapshot_download(repo_id=ADAPTER_FOLDER, cache_dir=_hf_hub_cache)
    model = PeftModel.from_pretrained(model, _adapter_local, is_trainable=False)

model.eval()
print("✓ Model ready")

## 6. Predict

Run inference on your reactions. Results are saved as a TSV file in `OUTPUT_DIR`.

In [ ]:
import types
from datasets import load_dataset
from src.evaluation.eval import evaluate_model

data = load_dataset("json", data_files={"pred": jsonl_path})

args = types.SimpleNamespace(
    output_folder=OUTPUT_DIR,
    eval_batch_size=8,          # reduce further if you run out of VRAM
    template_key=TEMPLATE_KEY,
    template_alias=TEMPLATE_ALIAS,
    model_id=MODEL_ID,
    few_shot_trained=False,
)

evaluate_model(
    accelerator, model, tokenizer,
    data["pred"], args,
    split="pred",
    save_predictions=True,
)

print(f"\n✓ Done! Predictions saved to: {OUTPUT_DIR}")

## 7. View results

The output file is named: `predictions_pred.EC_NUMBER.EC_NUMBER_PROMPT_2.<timestamp>.tsv`

It contains these columns:
- `pred_raw` — the raw EC number predicted by the model
- `pred_fragment` — leading digits/dots extracted (useful if the model outputs extra text)
- `pred_well_formed` — whether the prediction is a valid EC format (True/False)
- `true_label` — ground truth, if provided in the input CSV
- `smiles` — the input reaction SMILES

In [ ]:
import glob

tsv_files = glob.glob(f"{OUTPUT_DIR}/*.tsv")

if tsv_files:
    results = pd.read_csv(tsv_files[0], sep="\t")
    print(f"✓ {len(results)} predictions loaded from {tsv_files[0]}")
    display(results)
else:
    print(f"No TSV found in {OUTPUT_DIR} — check that the previous cell ran without errors.")